<div style="max-width:100%;box-sizing:border-box;overflow:visible;border-top:4px solid #0f766e;padding:32px 0 20px;margin:0 0 24px">
  <div style="display:block;color:#0f766e;font-size:13px;line-height:1.8;font-weight:700;letter-spacing:0.8px;text-transform:uppercase;margin:0 0 8px">LAB 9 · DATA WAREHOUSING WITH APACHE DORIS</div>
  <div style="color:#17212b;font-size:30px;line-height:1.3;font-weight:750;margin:0 0 10px">Order Facts, Customer Dimensions, and a Warehouse Chain</div>
</div>

Sales per customer need two tables: the orders, and the customers who placed them. You build `orders_fact_l2`, one row per order of Lab 7's current state, dated by the day the order was created, and `customers_dim_l2`, one row per WWI customer. You join them and prove, with row counts, distinct orders, amounts, and an anti-join, that the join neither lost nor doubled an order; read how the plan joins them; and watch a faulty dimension break both checks. Then you build a whole ODS → DWD → DWS → ADS chain on a small case of its own, which Labs 10 and 11 continue. Plan about 60 minutes.

**Module 9: Warehouse Modeling and Joins** · [Lesson](course.md) · [Quiz](quiz09_modeling_and_joins.ipynb) · [Course README](../../README.md)

## Before you begin

Complete Labs 6 and 7 first, in the same database. This Lab reads `customers` from Lab 6 and `orders_current` and `order_events` from Lab 7, and changes none of them. If one is missing, the setup cell stops and says which Lab to finish.

The Lab rebuilds `orders_fact_l2`, which Labs 10 and 12 read, and `customers_dim_l2`. Sections 5–7 rebuild the tables of the case: `ods_order_deliveries_l2`, `ods_payment_deliveries_l2`, `customers_case_dim_l2`, `dwd_orders_l2`, `dwd_payments_l2`, `dws_orders_region_day_l2`, and the view `ads_orders_region_day_l2`; Lab 10 continues from the case's DWD tables. All of them stay in your lab database. Everything runs in the course sandbox.

In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "dw_course").is_dir())
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from dw_course.docker_runtime import connect_sandbox
from dw_course.runtime import expect
from dw_course.ui import show_sql
from dw_course.warehouse import case_fixture

lab = connect_sandbox(module=9)

## 1. Declare the grain of the fact and the dimension

Before you join two tables, say what one row of each means: its **grain**. One row of `orders_fact_l2` is one order as it is now: Lab 7's current status, amounts, and region, dated by the event that created the order (version 1 in `order_events`), as Lab 8's view dates it. The fact keeps every order, the cancelled 900002 too: a report decides with its filter whether a cancelled order counts. One row of `customers_dim_l2` is one customer's current attributes, here the WWI customer's name, for all 663 customers of Lab 6.

The table models say how a second write of a key behaves. The fact is a Duplicate Key table: it keeps every row it is given, and nothing stops a second row for an order, so the cell checks the grain with `COUNT(*)` against `COUNT(DISTINCT order_id)`. The dimension is a Unique Key table on `customer_id`: loading a customer again replaces its row, so the dimension keeps one row per customer.

In [ ]:
lab.execute("DROP TABLE IF EXISTS orders_fact_l2")
lab.execute("DROP TABLE IF EXISTS customers_dim_l2")
dimension_ddl = """CREATE TABLE customers_dim_l2 (
    customer_id BIGINT NOT NULL,
    customer_name VARCHAR(100) NOT NULL
)
UNIQUE KEY(customer_id)
DISTRIBUTED BY HASH(customer_id) BUCKETS 1
PROPERTIES ("replication_num"="1")"""
fact_ddl = """CREATE TABLE orders_fact_l2 (
    order_date DATE NOT NULL,
    order_id BIGINT NOT NULL,
    customer_id BIGINT NOT NULL,
    status VARCHAR(20) NOT NULL,
    order_amount DECIMAL(12,2) NOT NULL,
    paid_amount DECIMAL(12,2) NOT NULL,
    refund_amount DECIMAL(12,2) NOT NULL,
    region VARCHAR(16) NOT NULL,
    data_source VARCHAR(32) NOT NULL
)
DUPLICATE KEY(order_date, order_id)
DISTRIBUTED BY HASH(customer_id) BUCKETS 1
PROPERTIES ("replication_num"="1")"""
load_dimension = "INSERT INTO customers_dim_l2 SELECT customer_id, customer_name FROM customers"
load_fact = """INSERT INTO orders_fact_l2
SELECT DATE(e.event_time), o.order_id, o.customer_id, o.status, o.order_amount,
       o.paid_amount, o.refund_amount, o.region, o.data_source
FROM orders_current o
JOIN order_events e ON e.order_id = o.order_id AND e.event_version = 1"""
show_sql("The dimension, the fact, and their loads", ";\n\n".join([dimension_ddl, fact_ddl, load_dimension, load_fact]) + ";")
for statement in (dimension_ddl, fact_ddl, load_dimension, load_fact):
    lab.execute(statement)
lab.sql("SELECT * FROM orders_fact_l2 ORDER BY order_id", title="The order fact: one row per order")
expect(lab.query("SELECT COUNT(*), COUNT(DISTINCT order_id), SUM(order_amount) FROM orders_fact_l2"),
       [(11, 11, "1510.00")], title="Eleven rows, eleven orders, 1510.00")
expect(lab.query("SELECT COUNT(*), COUNT(DISTINCT customer_id) FROM customers_dim_l2"), [(663, 663)],
       title="663 customers, each once")

**Expected result**

The two table definitions and their loads, then eleven orders: ten with `order_date` 2026-01-01 and 900011 with 2026-01-02; 900001 DELIVERED, 900002 CANCELLED, 900003 REFUNDED, the others CREATED; customers 1 to 11; region EAST or WEST. Two green checks: 11 rows for 11 orders and 1510.00, and 663 customers, each once.

The fact keeps 1510.00, the cancelled order's 200.00 included; Lab 8's definition, without cancelled orders, gives 1310.00. Both are right for their question, as long as each report says which one it uses.

## 2. Join, look for missing customers, and check the amounts

A `LEFT JOIN` keeps every order: an order whose customer is missing from the dimension still appears, with `NULL` in the customer's columns. The second query keeps only those orders: an anti-join, written as a left join with `WHERE d.customer_id IS NULL`. The third compares rows, distinct orders, and the amount before and after the join. Do not check missing customers with an inner join: it removes exactly the orders that you are looking for.

In [ ]:
lab.sql("""SELECT f.order_id, f.order_date, f.status, f.order_amount, d.customer_name
FROM orders_fact_l2 f
LEFT JOIN customers_dim_l2 d ON d.customer_id = f.customer_id
ORDER BY f.order_id""", title="Each order with its customer")
lab.sql("""SELECT f.customer_id, COUNT(*) AS orders_without_customer
FROM orders_fact_l2 f
LEFT JOIN customers_dim_l2 d ON d.customer_id = f.customer_id
WHERE d.customer_id IS NULL
GROUP BY f.customer_id""", title="Orders whose customer is missing")
before_after = """SELECT 'before the join' AS stage, COUNT(*) AS row_count,
       COUNT(DISTINCT order_id) AS order_count, SUM(order_amount) AS amount
FROM orders_fact_l2
UNION ALL
SELECT 'after the left join', COUNT(*), COUNT(DISTINCT f.order_id), SUM(f.order_amount)
FROM orders_fact_l2 f LEFT JOIN customers_dim_l2 d ON d.customer_id = f.customer_id
ORDER BY stage DESC"""
lab.sql(before_after, title="Rows, orders, and amount before and after the join")
expect(lab.query(before_after), [("before the join", 11, 11, "1510.00"), ("after the left join", 11, 11, "1510.00")],
       title="The join kept every order once")

**Expected result**

Eleven orders, each with its WWI customer's name, such as `Tailspin Toys (Head Office)` for customer 1. An empty table: no order lacks its customer. Then 11 rows, 11 orders, and 1510.00 both before and after the join, and a green check.

The three numbers check different things. The rows would grow if a customer matched twice; the distinct orders would fall if an inner join dropped an order; and the amount would change in either case, unless the two errors cancelled out, which the next steps show they can do.

## 3. Aggregate by region, and read how the plan joins

The region is recorded on each order, the sales region where it was booked. After the join, the totals per region must still add up to the 11 orders and 1510.00 of the fact; the check compares them.

The logical join says which rows appear; the plan says how Doris brings the matching rows together. `lab.explain_join()` shows each join of the plan with its type and its distribution: `BROADCAST` sends one side, usually the smaller one, to every instance that scans the other side; `PARTITIONED` shuffles both sides by the join key; `BUCKET_SHUFFLE` and `COLOCATE` use how the tables are already bucketed. The course sandbox has one BE, so the plan is for reading, not for measuring.

In [ ]:
by_region = """SELECT f.region, COUNT(*) AS order_count, COUNT(DISTINCT d.customer_id) AS customers, SUM(f.order_amount) AS amount
FROM orders_fact_l2 f
JOIN customers_dim_l2 d ON d.customer_id = f.customer_id
GROUP BY f.region
ORDER BY f.region"""
lab.sql(by_region, title="Orders and customers per region")
expect(lab.query(by_region), [("EAST", 6, 6, "830.00"), ("WEST", 5, 5, "680.00")], title="The regions add up to the fact")
joins = lab.explain_join("""SELECT f.order_id, d.customer_name
FROM orders_fact_l2 f JOIN customers_dim_l2 d ON d.customer_id = f.customer_id""", title="How the plan joins the order fact and the customers")
expect([join["join"] for join in joins], ["INNER JOIN"], title="One inner join")

**Expected result**

EAST with 6 orders, 6 customers, and 830.00, and WEST with 5 orders, 5 customers, and 680.00; together the fact's 11 orders and 1510.00, and the check is green. Then the plan: one `INNER JOIN` with the distribution `BROADCAST` and the condition on `customer_id`; the two scans, of `orders_fact_l2` and `customers_dim_l2`; and a green check.

`BROADCAST` sends the customers to the instance that scans the orders. On a cluster, the same plan would copy them to every instance that scans orders: cheap for a small dimension, expensive for a large one. The plan is Doris's choice for this data and its statistics; a Query Profile shows what the execution actually did.

## 4. Make a faulty join visible before you fix it

The dimension above has every customer once. Predict what two faults do. The query below joins the orders with a faulty copy of the dimension, written as a CTE, so `customers_dim_l2` itself stays as it is: customer 2 is missing, and customer 1 appears twice.

- A `LEFT JOIN` keeps customer 2's order, with `NULL` customer columns, and matches customer 1's order twice.
- An `INNER JOIN` drops customer 2's order and still matches customer 1's order twice.

Compare the joined rows, the distinct orders, the joined amount, and the unmatched orders. One total alone can hide two errors that offset each other.

In [ ]:
flawed_dimension = """
WITH flawed_dim AS (
    SELECT customer_id, customer_name
    FROM customers_dim_l2 WHERE customer_id <> 2
    UNION ALL
    SELECT customer_id, customer_name
    FROM customers_dim_l2 WHERE customer_id = 1
)
"""

def join_diagnostic(join_type):
    return flawed_dimension + f"""
SELECT COUNT(*) AS joined_rows,
       COUNT(DISTINCT f.order_id) AS distinct_orders,
       SUM(f.order_amount) AS joined_amount,
       SUM(CASE WHEN d.customer_id IS NULL THEN 1 ELSE 0 END) AS unmatched_orders
FROM orders_fact_l2 AS f
{join_type} JOIN flawed_dim AS d ON d.customer_id = f.customer_id
"""

lab.sql(join_diagnostic("LEFT"), title="The faulty dimension, left join")
lab.sql(join_diagnostic("INNER"), title="The faulty dimension, inner join")
expect(lab.query(join_diagnostic("LEFT")), [(12, 11, "1610.00", 1)], title="The left join keeps the unmatched order and doubles another")
expect(lab.query(join_diagnostic("INNER")), [(11, 10, "1410.00", 0)], title="The inner join drops the unmatched order and doubles another")
expect(lab.query("SELECT COUNT(*), SUM(order_amount) FROM orders_fact_l2"), [(11, "1510.00")], title="The fact is unchanged")

**Expected result**

The left join: 12 rows, 11 distinct orders, 1610.00, and 1 unmatched order. The inner join: 11 rows, 10 distinct orders, 1410.00, and no unmatched order. Three green checks; the last one says that the fact still has 11 orders and 1510.00.

In the left join, customer 2's order of 200.00 is still there with `NULL` customer columns, and customer 1's order of 100.00 counts twice: 12 rows and 100.00 too much. In the inner join, customer 2's order is gone and customer 1's still counts twice: 11 rows, the fact's own number, which looks right, but 10 distinct orders and 1410.00 show both errors. No plan could tell you this; only the checks of the result do.

## 5. ODS: keep every delivery of the case

The orders of Level 1 were created on two days and paid twice, too few to measure payments in a reporting window or customers who buy again. Labs 9 to 11 therefore follow a small case of their own, `datasets/warehouse_case.json`: 8 orders of 5 customers over three days, with deliveries that repeat, arrive late, or carry a malformed amount, and payments and a partial refund. It is a separate data set; its amounts are not Level 1's.

The warehouse layers give each table one job. ODS (Operational Data Store) keeps what the sources sent, one row per delivery, duplicates included; DWD (Data Warehouse Detail) keeps the validated detail at the business grain; DWS (Data Warehouse Summary) keeps aggregates that reports reuse; ADS (Application Data Service) is what a consumer reads. A layer is a modeling convention, not a kind of Doris table. The order fact and the customer dimension of steps 1–3 are Level 1's data at the DWD layer.

The cell drops the case's tables and loads the deliveries as they came: 10 order deliveries and 7 payment deliveries.

In [ ]:
case = case_fixture()
lab.execute("DROP VIEW IF EXISTS ads_orders_region_day_l2")
for table in ("dws_orders_region_day_l2", "dwd_orders_l2", "dwd_payments_l2", "customers_case_dim_l2", "ods_order_deliveries_l2", "ods_payment_deliveries_l2"):
    lab.execute(f"DROP TABLE IF EXISTS {table}")
lab.execute("""
CREATE TABLE ods_order_deliveries_l2 (
    input_id BIGINT NOT NULL, order_id BIGINT NOT NULL, customer_id BIGINT NOT NULL,
    created_at DATETIME NOT NULL, amount_text VARCHAR(40), status VARCHAR(20), event_version BIGINT NOT NULL
) DUPLICATE KEY(input_id) DISTRIBUTED BY HASH(input_id) BUCKETS 1 PROPERTIES ("replication_num"="1")
""")
lab.execute("""
CREATE TABLE ods_payment_deliveries_l2 (
    delivery_id BIGINT NOT NULL, payment_id VARCHAR(32) NOT NULL, order_id BIGINT NOT NULL,
    amount DECIMAL(18,2) NOT NULL, payment_time DATETIME NOT NULL
) DUPLICATE KEY(delivery_id) DISTRIBUTED BY HASH(delivery_id) BUCKETS 1 PROPERTIES ("replication_num"="1")
""")
lab.insert("ods_order_deliveries_l2", ["input_id","order_id","customer_id","created_at","amount_text","status","event_version"], case["order_deliveries"])
lab.insert("ods_payment_deliveries_l2", ["delivery_id","payment_id","order_id","amount","payment_time"], case["payment_deliveries"])
expect(lab.query("SELECT COUNT(*) FROM ods_order_deliveries_l2"), [(10,)], title="Ten order deliveries, kept as they came")
expect(lab.query("SELECT COUNT(*) FROM ods_payment_deliveries_l2"), [(7,)], title="Seven payment deliveries, the retry included")
lab.sql("SELECT input_id, order_id, amount_text, status, event_version FROM ods_order_deliveries_l2 ORDER BY input_id", title="The order deliveries in ODS");

**Expected result**

Two green checks, 10 order deliveries and 7 payment deliveries, then the order deliveries: order 2102 arrives twice, as delivery 2 with version 3 and `PARTIALLY_REFUNDED`, and later as delivery 9 with the older version 1 and `CREATED`; delivery 10, order 2199, has the amount `not-a-number`.

ODS keeps both, and the text of the bad amount: a later fix of a rule can reprocess them from here.

## 6. DWD: clean by business key, not by delivery

Predict: nine of the ten deliveries have a valid amount, and they hold eight distinct orders. The older delivery of 2102 must not turn it back into CREATED, and the malformed amount stays in ODS without becoming an order.

The query keeps one delivery per order, the one with the highest version, with `ROW_NUMBER()`: a window function that numbers the rows of each order (`PARTITION BY order_id`) in the order given (`ORDER BY event_version DESC, input_id DESC`), so the latest version gets number 1. Module 10 explains window functions. The payments keep one row per `payment_id`: the case says that a repeated payment ID carries the same content, so a retry is not a second payment; a different amount under the same ID would need a conflict rule of its own.

In [ ]:
lab.execute("""
CREATE TABLE customers_case_dim_l2 (customer_id BIGINT NOT NULL, region VARCHAR(16) NOT NULL)
UNIQUE KEY(customer_id) DISTRIBUTED BY HASH(customer_id) BUCKETS 1 PROPERTIES ("replication_num"="1")
""")
lab.insert("customers_case_dim_l2", ["customer_id","region"], case["customers"])
lab.execute("""
CREATE TABLE dwd_orders_l2 (
    order_id BIGINT NOT NULL, customer_id BIGINT NOT NULL, created_at DATETIME NOT NULL,
    order_amount DECIMAL(18,2) NOT NULL, status VARCHAR(20), event_version BIGINT NOT NULL
) UNIQUE KEY(order_id) DISTRIBUTED BY HASH(order_id) BUCKETS 1
PROPERTIES ("replication_num"="1", "function_column.sequence_col"="event_version")
""")
lab.execute("""
INSERT INTO dwd_orders_l2
SELECT order_id, customer_id, created_at, CAST(amount_text AS DECIMAL(18,2)), status, event_version
FROM (
    SELECT *, ROW_NUMBER() OVER(PARTITION BY order_id ORDER BY event_version DESC, input_id DESC) AS rn
    FROM ods_order_deliveries_l2
    WHERE amount_text REGEXP '^[0-9]+([.][0-9]{1,2})?$'
) valid WHERE rn=1
""")
lab.execute("""
CREATE TABLE dwd_payments_l2 (
    payment_id VARCHAR(32) NOT NULL, order_id BIGINT NOT NULL,
    amount DECIMAL(18,2) NOT NULL, payment_time DATETIME NOT NULL
) UNIQUE KEY(payment_id) DISTRIBUTED BY HASH(payment_id) BUCKETS 1 PROPERTIES ("replication_num"="1")
""")
lab.execute("""
INSERT INTO dwd_payments_l2
SELECT payment_id, order_id, amount, payment_time FROM (
    SELECT *, ROW_NUMBER() OVER(PARTITION BY payment_id ORDER BY delivery_id DESC) AS rn
    FROM ods_payment_deliveries_l2
) payments WHERE rn=1
""")
expect(lab.query("SELECT COUNT(*), SUM(order_amount) FROM dwd_orders_l2"), [(8, "850.00")], title="Eight current orders, 850.00, the cancelled one included")
expect(lab.query("SELECT status, event_version FROM dwd_orders_l2 WHERE order_id=2102"), [("PARTIALLY_REFUNDED", 3)], title="The late older delivery does not win")
expect(lab.query("SELECT COUNT(*) FROM dwd_payments_l2"), [(6,)], title="Six payments: the retry is not another payment")
expect(lab.query("SELECT o.order_id FROM dwd_orders_l2 o LEFT JOIN customers_case_dim_l2 c ON o.customer_id=c.customer_id WHERE c.customer_id IS NULL"), [], title="Every order has its customer")
lab.sql("SELECT order_id, customer_id, order_amount, status, event_version FROM dwd_orders_l2 ORDER BY order_id", title="The current orders in DWD");

**Expected result**

Four green checks: 8 current orders and 850.00, order 2102 `PARTIALLY_REFUNDED` at version 3, 6 payments, and every order with its customer. Then the eight orders, 2101 to 2108, with their status and version; 2107 is CANCELLED.

The DWD orders keep the cancelled 2107 and its 50.00, as the order fact of step 1 keeps 900002: the detail records what happened, and the next layer applies the report's rule.

## 7. DWS and ADS: aggregate by the contract, publish, and reconcile

The summary counts orders and their amount per order date and region, by the same rule as Lab 8's view: cancelled orders are not counted. The amount is the order amount, not the money paid: Lab 10 measures payments from `dwd_payments_l2`, and a partial refund does not change an order's amount. The ADS view publishes the summary's rows under the names that a dashboard reads.

Predict: five rows, one per date and region that has orders, holding 7 orders. The cell checks them against the case's own expected rows, and the totals against the DWD detail.

In [ ]:
lab.execute("""
CREATE TABLE dws_orders_region_day_l2 (
    order_date DATE NOT NULL, region VARCHAR(16) NOT NULL,
    order_count BIGINT NOT NULL, order_amount DECIMAL(18,2) NOT NULL
) DUPLICATE KEY(order_date, region)
DISTRIBUTED BY HASH(order_date) BUCKETS 1 PROPERTIES ("replication_num"="1")
""")
lab.execute("""
INSERT INTO dws_orders_region_day_l2
SELECT DATE(o.created_at), c.region, COUNT(*), SUM(o.order_amount)
FROM dwd_orders_l2 o JOIN customers_case_dim_l2 c ON o.customer_id=c.customer_id
WHERE o.status <> 'CANCELLED'
GROUP BY DATE(o.created_at), c.region
""")
lab.execute("""
CREATE VIEW ads_orders_region_day_l2 AS
SELECT order_date, region, order_count, order_amount FROM dws_orders_region_day_l2
""")
expect(lab.query("SELECT CAST(order_date AS STRING), region, order_count, order_amount FROM ads_orders_region_day_l2 ORDER BY order_date, region"), case["expected"]["order_daily"], title="The published rows match the case's expected rows")
expect(lab.query("SELECT SUM(order_count), SUM(order_amount) FROM ads_orders_region_day_l2"),
       lab.query("SELECT COUNT(*), SUM(order_amount) FROM dwd_orders_l2 WHERE status <> 'CANCELLED'"),
       title="ADS adds up to the DWD orders that count: 7 orders, 800.00")
lab.sql("SELECT * FROM ads_orders_region_day_l2 ORDER BY order_date, region", title="The published orders per day and region");

**Expected result**

Two green checks, then five rows: 2026-01-01 EAST with 2 orders and 180.00, 2026-01-01 WEST with 1 and 120.00, 2026-01-02 EAST with 1 and 150.00, 2026-01-02 WEST with 1 and 200.00, and 2026-01-03 WEST with 2 and 150.00. Together 7 orders and 800.00.

The deliveries, the orders, and the published rows have different counts, 10, 8, and 5, because each has its own grain: one delivery, one order, one date and region. The amount ties them together: the 800.00 of the published rows is the DWD orders' 850.00 without the cancelled 2107's 50.00. Rebuilding these tables from ODS gives the same rows again; inserting the summary a second time would not, because the summary would then hold every order twice.

## 8. Practice: repair the faulty dimension

Write a new CTE, `repaired_dim`, that gives every order of the fact exactly one customer, and run the four checks of step 4 with both a left and an inner join. Expect 11 rows, 11 distinct orders, 1510.00, and no unmatched order from each. Do not change `customers_dim_l2` or the Level 1 tables: the CTE replaces `flawed_dim` only in your query.

In [ ]:
# Your code here

The next cell is a reference solution; it is collapsed. Expand it to compare, or run it.

In [ ]:
repaired_dim = """
WITH repaired_dim AS (
    SELECT customer_id, customer_name FROM customers_dim_l2
)
"""

def repaired_join(join_type):
    return repaired_dim + f"""
SELECT COUNT(*) AS joined_rows, COUNT(DISTINCT f.order_id) AS distinct_orders,
       SUM(f.order_amount) AS joined_amount,
       SUM(CASE WHEN d.customer_id IS NULL THEN 1 ELSE 0 END) AS unmatched_orders
FROM orders_fact_l2 AS f
{join_type} JOIN repaired_dim AS d ON d.customer_id = f.customer_id
"""

lab.sql(repaired_join("LEFT"), title="The repaired dimension, left join")
expect(lab.query(repaired_join("LEFT")), [(11, 11, "1510.00", 0)], title="The left join keeps every order once")
expect(lab.query(repaired_join("INNER")), [(11, 11, "1510.00", 0)], title="The inner join keeps every order once")

**Expected result**

The left join: 11 rows, 11 distinct orders, 1510.00, and no unmatched order; two green checks, for the left and the inner join.

The faulty CTE removed customer 2 and added a second row for customer 1. The repaired one takes the dimension as it is, one row per customer; a CTE that put back customer 2 and kept one row of customer 1 would pass the same checks. Both joins now agree with each other and with the fact, which is what makes the dimension right for this fact: its key is unique, and every order finds its customer.

## You are done when

- you can say what one row of `orders_fact_l2`, `customers_dim_l2`, `dwd_orders_l2`, and `ads_orders_region_day_l2` means, and why the fact counts 1510.00 and the reports 1310.00;
- the join of the fact and the dimension keeps 11 orders and 1510.00, the anti-join finds no missing customer, and you can read the join type and distribution in the plan;
- you can explain from rows, distinct orders, and amounts what the faulty dimension did to each join;
- the case's chain keeps 10 deliveries, 8 orders, and 5 published rows of 7 orders and 800.00, and you can say why the counts differ.

## Next

Open [Quiz 9: Warehouse Modeling and Joins](quiz09_modeling_and_joins.ipynb), then continue to [Module 10: Metric Processing and Delivery](../module10-metric-processing/course.md).